# 01 - Data preparation
Order: 1) missing values, 2) top 5 columns, 3) other quality issues, 4) cleaning. Results are printed as Markdown tables to copy into REPORT.md (sections 1.1 to 1.4). Cells 1 and 3 read the full dataset: run them once, they take several minutes.

In [1]:
import sys
sys.path.insert(0, "/home/jovyan")

from src.spark_session import get_spark
from src.loaders import read_raw
from src import quality
from src.reporting import show_md

spark = get_spark("preparation", driver_memory="4g")

## Step 1 - Missing values per column (section 1.1)

In [2]:
profile = quality.missing_report(spark, read_raw)
profile.cache()
show_md(profile, 100)

| table | column | total_rows | n_null | n_backslash_n | n_empty | n_missing | missing_pct |
|---|---|---|---|---|---|---|---|
| title.akas | attributes | 60424702 | 0 | 60107437 | 0 | 60107437 | 99.4749 |
| title.basics | endYear | 12963111 | 0 | 12798642 | 0 | 12798642 | 98.7313 |
| name.basics | deathYear | 15816005 | 0 | 15550475 | 0 | 15550475 | 98.3211 |
| name.basics | birthYear | 15816005 | 0 | 15130128 | 0 | 15130128 | 95.6634 |
| title.principals | job | 102756358 | 0 | 82882023 | 0 | 82882023 | 80.6588 |
| title.akas | types | 60424702 | 0 | 40393110 | 0 | 40393110 | 66.8487 |
| title.basics | runtimeMinutes | 12963111 | 0 | 8050574 | 0 | 8050574 | 62.1037 |
| title.principals | characters | 102756358 | 0 | 52691057 | 0 | 52691057 | 51.2777 |
| title.crew | writers | 12960686 | 0 | 6432432 | 0 | 6432432 | 49.6303 |
| title.crew | directors | 12960686 | 0 | 5791405 | 0 | 5791405 | 44.6844 |
| title.akas | language | 60424702 | 0 | 19665115 | 0 | 19665115 | 32.5448 |
| title.a

## Step 2 - Five columns with the highest missing percentage (section 1.2)
Ties are broken by table name then column name.

In [3]:
show_md(quality.top_missing(profile, 5))

| table | column | missing_pct |
|---|---|---|
| title.akas | attributes | 99.4749 |
| title.basics | endYear | 98.7313 |
| name.basics | deathYear | 98.3211 |
| name.basics | birthYear | 95.6634 |
| title.principals | job | 80.6588 |


## Step 3 - Other data quality issues (section 1.3)
Slow cells: duplicates on `title.principals` and the person/film check join several large tables.

In [ ]:
checks = quality.run_checks(spark, read_raw)
checks.cache()
show_md(checks, 100)

## Step 4 - Cleaning (section 1.4)
Writes `data/parquet/clean/<table>` and `data/parquet/quarantine/<table>`. Parents (`title.basics`, `name.basics`) are cleaned first.

In [ ]:
stats = quality.clean_all(spark, read_raw)
show_md(stats)

## Check of the clean tables

In [ ]:
from src.loaders import read_clean
for t in ["title.basics", "name.basics", "title.ratings"]:
    df = read_clean(spark, t)
    df.printSchema()
    df.show(3, truncate=False)